In [1]:
!pip install -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 6.9 MB/s eta 0:00:00


In [2]:
!curl -L -o fairface.zip\
  https://www.kaggle.com/api/v1/datasets/download/aibloy/fairface

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  549M  100  549M    0     0  77.0M      0  0:00:07  0:00:07 --:--:-- 82.2M


In [3]:
! unzip fairface.zip

Streaming output truncated to the last 5000 lines.
  inflating: FairFace/val/550.jpg    
  inflating: FairFace/val/5500.jpg   
  inflating: FairFace/val/5501.jpg   
  inflating: FairFace/val/5502.jpg   
  inflating: FairFace/val/5503.jpg   
  inflating: FairFace/val/5504.jpg   
  inflating: FairFace/val/5505.jpg   
  inflating: FairFace/val/5506.jpg   
  inflating: FairFace/val/5507.jpg   
  inflating: FairFace/val/5508.jpg   
  inflating: FairFace/val/5509.jpg   
  inflating: FairFace/val/551.jpg    
  inflating: FairFace/val/5510.jpg   
  inflating: FairFace/val/5511.jpg   
  inflating: FairFace/val/5512.jpg   
  inflating: FairFace/val/5513.jpg   
  inflating: FairFace/val/5514.jpg   
  inflating: FairFace/val/5515.jpg   
  inflating: FairFace/val/5516.jpg   
  inflating: FairFace/val/5517.jpg   
  inflating: FairFace/val/5518.jpg   
  inflating: FairFace/val/5519.jpg   
  inflating: FairFace/val/552.jpg    
  inflating: FairFace/val/5520.jpg   
  inflating: FairFace/val/5521.jpg   

In [4]:
import os
import shutil
import pandas as pd
from tqdm import tqdm


train_df = pd.read_csv("FairFace/train_labels.csv")
val_df = pd.read_csv("FairFace/val_labels.csv")

In [5]:
train_df["age"].unique()

array(['50-59', '30-39', '3-9', '20-29', '40-49', '10-19', '60-69', '0-2',
       'more than 70'], dtype=object)

In [6]:
train_df = train_df[~(train_df["age"].isin(['0-2', '3-9', '10-19']))]
val_df = val_df[~(val_df["age"].isin(['0-2', '3-9', '10-19']))]

train_df.age.unique()

array(['50-59', '30-39', '20-29', '40-49', '60-69', 'more than 70'],
      dtype=object)

In [7]:
def apply_adult_elderly_filter(age:str):
  if age in ["60-69", "more than 70"]:
    return "Elderly"
  return "Adult"

train_df["age"] = train_df["age"].apply(lambda age: apply_adult_elderly_filter(age))
val_df["age"] = val_df["age"].apply(lambda age: apply_adult_elderly_filter(age))

In [9]:
# Create the root directory and child folders
base_dir = 'FairFace-Age'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')

for age_cat in list(train_df["age"].unique()):
  path_train = os.path.join(train_dir, age_cat)
  path_val = os.path.join(val_dir, age_cat)
  os.makedirs(path_train, exist_ok=True)
  os.makedirs(path_val, exist_ok=True)

In [10]:
def organize_age_images(df, source_dir_prefix, target_dir_prefix):
  # Read labels CSV

    # Ensure age values are mapped correctly
    # The column in FairFace is usually 'age'
    age_col = 'age'
    file_col = 'file'

    print(f"Processing...")

    # Counter for copied files
    copied_count = 0

    for _, row in tqdm(df.iterrows(), total=len(df)):
        file_path = row[file_col]
        age = row[age_col]

        age_folder = age

        # Prepare source and target file paths
        src = os.path.join(source_dir_prefix, file_path)
        dest = os.path.join(target_dir_prefix, age_folder, os.path.basename(file_path))

        if os.path.exists(src):
            shutil.copy2(src, dest)
            copied_count += 1

    print(f"Successfully copied {copied_count} files to {target_dir_prefix}.")

# Process train dataset
organize_age_images(train_df, 'FairFace', train_dir)

# Process validation dataset
organize_age_images(val_df, 'FairFace', val_dir)

Processing...


100%|██████████| 65441/65441 [00:15<00:00, 4361.08it/s]


Successfully copied 65441 files to FairFace-Age/train.
Processing...


100%|██████████| 8218/8218 [00:01<00:00, 4636.33it/s]

Successfully copied 8218 files to FairFace-Age/val.


In [11]:
from ultralytics import YOLO

# Load a model
model = YOLO("age-cls.pt")  # load a pretrained model (recommended for training)

# Train the model
results = model.train(data="FairFace-Age", epochs=10, imgsz=128, batch=128, cls=2, cls_pw=1)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=2, cls_pw=1, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=FairFace-Age, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0

In [12]:
!zip -r runs.zip runs/
from google.colab import files
files.download('runs.zip')

  adding: runs/ (stored 0%)
  adding: runs/classify/ (stored 0%)
  adding: runs/classify/train/ (stored 0%)
  adding: runs/classify/train/train_batch1.jpg (deflated 5%)
  adding: runs/classify/train/results.csv (deflated 62%)
  adding: runs/classify/train/results.png (deflated 9%)
  adding: runs/classify/train/confusion_matrix.png (deflated 36%)
  adding: runs/classify/train/weights/ (stored 0%)
  adding: runs/classify/train/weights/last.pt (deflated 10%)
  adding: runs/classify/train/weights/best.pt (deflated 10%)
  adding: runs/classify/train/val_batch0_labels.jpg (deflated 2%)
  adding: runs/classify/train/args.yaml (deflated 52%)
  adding: runs/classify/train/confusion_matrix_normalized.png (deflated 37%)
  adding: runs/classify/train/val_batch0_pred.jpg (deflated 2%)
  adding: runs/classify/train/val_batch2_pred.jpg (deflated 2%)
  adding: runs/classify/train/val_batch1_pred.jpg (deflated 2%)
  adding: runs/classify/train/train_batch0.jpg (deflated 4%)
  adding: runs/classify/trai

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>